In [ ]:
#first import the relevant libraries
import pandas as pd
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve
)

import matplotlib.pyplot as plt

In [ ]:
#assign variables to the different processed data files
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [ ]:
#print the shape of the data
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
#build our first model, logistic regression using a maximum iteration of 1000
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

In [ ]:
y_pred = log_model.predict(X_test)

In [ ]:
y_prob = log_model.predict_proba(X_test)[:, 1]

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
#Prints a report of the perfomance metrics of this logistic regression model
print(classification_report(y_test, y_pred))

In [ ]:
roc = roc_auc_score(y_test, y_prob)

print(roc)

In [ ]:
pr_auc = average_precision_score(y_test, y_prob)

print(pr_auc)

In [ ]:
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

In [ ]:
xgb_model.fit(X_train, y_train)

In [ ]:
xgb_pred = xgb_model.predict(X_test)

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

In [ ]:
print(classification_report(y_test, xgb_pred))


In [ ]:
print(confusion_matrix(y_test, xgb_pred))

In [ ]:
print("ROC AUC:", roc_auc_score(y_test, xgb_prob))

In [ ]:
print("PR AUC:", average_precision_score(y_test, xgb_prob))

In [ ]:
import joblib
joblib.dump(
    xgb_model,
    "../models/xgb_fraud_model.pkl"
)

In [ ]:
model = joblib.load("../models/xgb_fraud_model.pkl")


In [ ]:
threshold = 0.30

y_pred_custom = (xgb_prob >= threshold).astype(int)

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    "n_estimators":[100,200],
    "max_depth":[3,5],
    "learning_rate":[0.5,0.1]
}

In [ ]:
grid = GridSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),
    param_grid=param_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1
)

In [ ]:
grid.fit(X_train, y_train)

In [ ]:
print(grid.best_params_)

In [ ]:
best_model = grid.best_estimator_

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
scores = cross_val_score(
    best_model,
    X_train,
    y_train,
    cv=5,
    scoring="average_precision"
)

In [ ]:
scores = cross_val_score(
    best_model,
    X_train,
    y_train,
    cv=5,
    scoring="average_precision"
)

In [ ]:
print(scores)

In [ ]:
print(scores.mean())

In [ ]:
import matplotlib.pyplot as plt

importance = best_model.feature_importances_

feature_names = X_train.columns

importance_df = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance
    })
    .sort_values("Importance", ascending=False)
)

print(importance_df.head(10))

In [ ]:
top10 = importance_df.head(10)

plt.figure(figsize=(8,6))
plt.barh(top10["Feature"], top10["Importance"])
plt.gca().invert_yaxis()
plt.title("Top 10 Most Important Features")
plt.xlabel("Importance")
plt.show()

In [ ]:
#Instead of using the default threshold, we can use several and loop through them
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

for t in thresholds:
    predictions = (best_model.predict_proba(X_test)[:, 1] >= t).astype(int)

    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    print(f"Threshold: {t}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"F1: {f1:.3f}")
    print()

In [ ]:
#There isn't a universally "best" threshold—it depends on the business objective.

#Banks often prefer high recall, because missing a fraudulent transaction can be very costly.